In [1]:
import pandas as pd
import numpy as np
import json
import os
import ast
from pathlib import Path
import torch
from typing import List, Optional
from dataclasses import dataclass

import teradatasql
from sqlalchemy import text, create_engine
from teradataml import create_context, get_context, get_connection, DataFrame, in_schema, copy_to_sql
from teradataml.dataframe.copy_to import copy_to_sql
from dotenv import load_dotenv

import torch
from sentence_transformers import SentenceTransformer

In [2]:
#from config.settings import (TD_HOST, TD_USER, TD_PASS, TD_DB)
TD_HOST="iteration7-w9og53takluu3v27.env.clearscape.teradata.com"
TD_USER="demo_user"
TD_PASS="n8888888"
TD_DB="DEMO_USER"

In [3]:
conn = teradatasql.connect(
    host=TD_HOST,
    user=TD_USER,
    password=TD_PASS,
    logdata={'CHARSET': 'UTF8'}
)

sqlalchemy_engine = create_engine("teradatasql://", creator=lambda: conn)
create_context(tdsqlengine=sqlalchemy_engine)
print("Connection successful with UTF-8 encoding!")

Connection successful with UTF-8 encoding!


# cleaning

In [4]:
tdf = DataFrame.from_table("original_dataset", schema_name=TD_DB, index_label="Item_Name")
print("Shape of the data:", tdf.shape)

Shape of the data: (4773, 10)


In [5]:
tdf.head(10)

Item_Name,class,Brand,Weight,Number of units,Size of units,Price,T.Price,Pack,Unit
أبو كاس أرز مزة بسمتي هندي 10 كجم,"Rice, Pasta & Pulses",أبو كاس,10كجم,1,None,None,None,كيس,كجم
أجنحة دجاج أطياب - 700جم,Poultry,أطياب,700جم,1,None,None,None,عبوة,جم
أجنحة دجاج حارة أطياب,Poultry,أطياب,None,1,None,None,None,عبوة,None
أحمد تي شاي أخضر نقي - 20 فتلة,"Tea, Coffee & Hot Drinks",أحمد تي,None,20,None,None,None,عبوة,None
أحمد تي شاي إيرل جراي - 25 فتلة,"Tea, Coffee & Hot Drinks",أحمد تي,None,25,None,None,None,عبوة,None
أحمد تي كولد برو شاي مثلج بالليمون والنعناع 20 كيس,"Tea, Coffee & Hot Drinks",احمد تي,None,20,None,None,None,None,كيس
أحمد تي شاي إيرل جراي - 100 فتلة,"Tea, Coffee & Hot Drinks",أحمد تي,100جم,1,None,None,None,علبة,جم
أجرومونتي صلصة الداترينو 330 جم,"Tins, Jars & Packets",أجرومونتي,330جم,1,None,None,None,عبوة,جم
أبو عوف قهوة تركي محوج وسط 250 جم,"Tea, Coffee & Hot Drinks",أبو عوف,250جم,1,None,None,None,عبوة,جم
أبو علي بابريكا - 85جم,Cooking Ingredients,أبو علي,85جم,1,None,None,None,عبوة,جم


In [6]:
tdf = tdf.dropna(subset=["Item_Name", "class"])

In [7]:
tdf = tdf.assign(
    Item_Name = tdf.Item_Name.str.lower(),
    **{'class': tdf['class'].str.lower()}
)

In [8]:
tdf_stripped = tdf.assign(
    Item_Name = tdf.Item_Name.str.strip(),
    **{'class': tdf['class'].str.strip()}
)

*Embeddings*

In [ ]:
#product embeddings#
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import torch

INPUT_CSV  = "data/cleaned_test.csv"
OUTPUT_CSV = "outputs/ccleaned_test_embeddings.csv"
MODEL_NAME = "intfloat/multilingual-e5-large-instruct"

# Load data
df = pd.read_csv(INPUT_CSV)
texts = df["cleaned_text"].fillna("").astype(str).tolist()

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)

# 👉 prefix as passage
texts_for_model = [f"passage: {t}" for t in texts]

# Embed
emb = model.encode(
    texts_for_model,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)
dim = emb.shape[1]

# Save
emb_cols = [f"v{i}" for i in range(1, dim + 1)]
out = pd.concat(
    [df.reset_index(drop=True), pd.DataFrame(emb, columns=emb_cols)], axis=1
)

if "row_id" not in out.columns:
    out.insert(0, "row_id", np.arange(1, len(out) + 1, dtype=np.int64))

out.to_csv(OUTPUT_CSV, index=False)


print(f"✅ Saved {len(out)} rows with {dim}-dim passage embeddings → {OUTPUT_CSV}")

*cosine similarity*

In [ ]:
import teradatasql
import pandas as pd
from config.settings import TD_HOST, TD_USER, TD_PASS, TD_DB

ITEM_EMB  = f"{TD_DB}.products_labels_fc"     # has: row_id, v1..vN
REF_EMB   = f"{TD_DB}.original_labels_fc"     # has: <some id>, v1..vN
RESULT    = f"{TD_DB}.original_label_predictions_fc"

def get_columns(con, db, table):
    sql = f"""
    SELECT ColumnName
    FROM DBC.ColumnsV
    WHERE DatabaseName = '{db}'
      AND TableName    = '{table.split('.')[-1]}'
    ORDER BY ColumnId
    """
    return pd.read_sql(sql, con)["ColumnName"].tolist()

def pick_ref_id(colnames):
    for c in ["gpc_id", "label_id", "orig_label_id", "id", "row_id"]:
        if c in colnames:
            return c
    raise ValueError(f"No suitable ID column in {REF_EMB}. Found: {colnames}")

with teradatasql.connect(host=TD_HOST, user=TD_USER, password=TD_PASS) as con:
    cur = con.cursor()

    item_cols = get_columns(con, TD_DB, ITEM_EMB)
    ref_cols  = get_columns(con, TD_DB, REF_EMB)

    if "row_id" not in item_cols:
        raise ValueError(f"{ITEM_EMB} must contain 'row_id'. Found: {item_cols}")

    ref_id_col = pick_ref_id(ref_cols)  # could be 'row_id' in your case
    # output alias for the reference id — force a DIFFERENT name than 'row_id'
    ref_id_out = "orig_label_id"               # choose what you like: 'gpc_id'/'orig_label_id'/...

    # ensure shared v* features & consistent order
    item_feats = [c for c in item_cols if c.startswith("v")]
    ref_feats  = [c for c in ref_cols  if c.startswith("v")]
    feats      = sorted(set(item_feats).intersection(ref_feats),
                        key=lambda x: int(x[1:]) if x[1:].isdigit() else x)
    if not feats:
        raise ValueError("No shared v* feature columns across both tables.")

    vec_cols         = ", ".join(feats)
    vec_cols_quoted  = ", ".join(f"'{c}'" for c in feats)

    # (re)create result table – output columns: row_id, gpc_id, score
    try:
        cur.execute(f"DROP TABLE {RESULT};")
    except Exception:
        pass

    create_sql = f"""
    CREATE MULTISET TABLE {RESULT} AS
    (
      SELECT
        o.Target_ID    AS row_id,
        o.Reference_ID AS {ref_id_out},
        1 - o.Distance AS score
      FROM TD_SYSFNLIB.TD_VectorDistance
      (
        ON (SELECT row_id, {vec_cols} FROM {ITEM_EMB}) AS TargetTable
        ON (SELECT {ref_id_col}, {vec_cols} FROM {REF_EMB})  AS ReferenceTable DIMENSION
        USING
          TargetIDColumn       ('row_id')
          RefIDColumn          ('{ref_id_col}')
          TargetFeatureColumns ({vec_cols_quoted})
          RefFeatureColumns    ({vec_cols_quoted})
          DistanceMeasure      ('cosine')
      ) AS o
      QUALIFY ROW_NUMBER() OVER (PARTITION BY o.Target_ID ORDER BY o.Distance) = 1
    ) WITH DATA
    PRIMARY INDEX (row_id);
    """
    cur.execute(create_sql)

print(f"✅ Rebuilt {RESULT} with top‑1 cosine similarity. Columns: row_id, {ref_id_out}, score")


*eval*

In [ ]:
import teradatasql
import pandas as pd

TD_HOST="iteration7-w9og53takluu3v27.env.clearscape.teradata.com"
TD_USER="demo_user"
TD_PASS="n8888888"

LABELS_CLAUSE = """
Labels(
 'Condiments, Dressings & Marinades','Furniture','Personal care, skin & body care','null',
 'Tea, Coffee & Hot Drinks','Sweets & Desserts','Hair, Shower, Bath & Soap','Fruits',
 'Nuts, Dates & Dried Fruits','Vegetables & Fruits','Home Appliances',
 'Sauces, Dressings & Condiments','Baby Care','Tea and Coffee','Disposables & Napkins',
 'Tins, Jars & Packets','Chips & Crackers','Soft Drinks & Juices','Cooking Ingredients',
 'Dairy & Eggs','Bakery','Vegetables & Herbs','Biscuits & Cakes','Candles & Air Fresheners',
 'Water','Rice, Pasta & Pulses','Poultry','Beef & Processed Meat','Home Textile',
 'Cleaning Supplies','Beef & Lamb Meat','Chocolates, Sweets & Desserts','Jams, Spreads & Syrups'
)
"""

with teradatasql.connect(host=TD_HOST, user=TD_USER, password=TD_PASS) as con:
    cur = con.cursor()

    # 0) (Optional) sanity probe of column names
    probe = """
    SEL TableName, ColumnId, ColumnName
    FROM DBC.ColumnsV
    WHERE DatabaseName='demo_user'
      AND TableName IN ('cleaned_final_with_id','original_labels_lookup','original_label_predictions_fc')
    ORDER BY TableName, ColumnId;
    """
    print(pd.read_sql(probe, con))

    # 1) Drop & create results table explicitly (avoid CTAS parser edge cases)
    try:
        cur.execute("DROP TABLE demo_user.results;")
    except Exception as e:
        if "3807" not in str(e):
            raise

    cur.execute("""
    CREATE MULTISET TABLE demo_user.results
    (
      actual_class    VARCHAR(512),
      predicted_class VARCHAR(512)
    );
    """)
    print("✅ Created demo_user.results")

    # 2) Populate results via INSERT ... SELECT
    insert_sql = """
    INSERT INTO demo_user.results (actual_class, predicted_class)
    SELECT
        cf."class"      AS actual_class,
        lkp."class"     AS predicted_class
    FROM demo_user.original_label_predictions_fc p
    JOIN demo_user.cleaned_final_with_id cf
        ON cf.row_id = p.row_id
    JOIN demo_user.original_labels_lookup lkp
        ON lkp.row_id = p.gpc_id;   -- your lookup uses 'row_id' as the label id
    """
    cur.execute(insert_sql)
    print(pd.read_sql("SEL COUNT(*) AS n FROM demo_user.results;", con))

    # 3) Persist evaluator output via CTAS (no VOLATILE, no OUT TABLE syntax)
    try:
        cur.execute("DROP TABLE demo_user.classification_metrics;")
    except Exception as e:
        if "3807" not in str(e):
            raise

    eval_ctas = f"""
    CREATE MULTISET TABLE demo_user.classification_metrics AS
    (
      SELECT *
      FROM TD_ClassificationEvaluator (
         ON demo_user.results AS InputTable
         USING
             ObservationColumn('actual_class')
             PredictionColumn('predicted_class')
             {LABELS_CLAUSE}
      ) AS dt
    ) WITH DATA;
    """
    cur.execute(eval_ctas)
    print("✅ Created demo_user.classification_metrics")

    # 4) Fetch metrics (persistent table)
    df = pd.read_sql("SELECT * FROM demo_user.classification_metrics;", con)

print("✅ Columns:", list(df.columns))

# Optional: sort by best-guess column names if present
metric_cols = ["metric_name", "Metric", "metric"]
label_cols  = ["class_label", "ClassLabel", "label", "Label"]

metric_col = next((c for c in metric_cols if c in df.columns), None)
label_col  = next((c for c in label_cols  if c in df.columns), None)

if metric_col and label_col:
    df = df.sort_values([metric_col, label_col])
elif metric_col:
    df = df.sort_values([metric_col])

print("\n🔎 Head:")
print(df.head(20))

*model loading*

In [1]:
%pip -q install "transformers>=4.39" "accelerate>=0.26" "bitsandbytes>=0.41" safetensors einops

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install -U pip
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

  Obtaining dependency information for pip from https://files.pythonhosted.org/packages/b7/3f/945ef7ab14dc4f9d7f40288d2df998d1837ee0888ec3659c813487572faa/pip-25.2-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8

  You can safely remove it manually.



   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4 [torch]
   ---------- ----------------------------- 1/4

In [3]:
import torch, platform
import transformers, bitsandbytes
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| CC:", torch.cuda.get_device_capability(0))
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

Torch: 2.5.1+cu121 | CUDA available: True
GPU: NVIDIA RTX 2000 Ada Generation Laptop GPU | CC: (8, 9)
Transformers: 4.55.2
bitsandbytes: 0.47.0


In [4]:
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig

model_id = "Qwen/Qwen3-Embedding-8B"

# Pick compute dtype for GEMMs (weights stay 4-bit)
supports_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16

# Tokenizer
tok = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Quantization config (runtime, weight-only 4-bit)
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

# Load model quantized at load-time
mdl = AutoModel.from_pretrained(
    model_id,
    trust_remote_code=True,
    device_map="auto",                 # let HF place layers on your GPU
    quantization_config=bnb_cfg,       # <-- triggers runtime 4-bit
    torch_dtype=compute_dtype,         # compute dtype; weights remain 4-bit
    # If you hit OOM, uncomment the two lines below and try again:
    # max_memory={0: "7GiB", "cpu": "30GiB"},
    # offload_state_dict=True,
).eval()

# Verify 4-bit is really active
try:
    from bitsandbytes.nn import Linear4bit
    is_4bit = any(isinstance(m, Linear4bit) for m in mdl.modules())
    print("Using 4-bit quantized layers:", is_4bit)
except Exception as e:
    print("bitsandbytes verification failed:", e)

print("Loaded device:", next(mdl.parameters()).device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\na255073\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\na255073\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-8B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

model-00002-of-00004.safetensors:  67%|######7   | 3.31G/4.92G [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/Qwen/Qwen3-Embedding-8B/resolve/1d8ad4ca9b3dd8059ad90a75d4983776a23d44af/model-00001-of-00004.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...


model-00001-of-00004.safetensors:  83%|########3 | 4.07G/4.90G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Using 4-bit quantized layers: True
Loaded device: cuda:0


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, os

model_id = "tiiuae/falcon-7b" 

# Pick compute dtype for GEMMs (weights stay 4-bit)
supports_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=True,
    trust_remote_code=True  # Falcon uses custom code
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    # --- GPU path: 4-bit weight-only quantization at LOAD TIME ---
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        quantization_config=bnb_cfg,      # ← triggers runtime 4-bit quantization
        torch_dtype=compute_dtype,        # compute dtype only; weights are 4-bit
        trust_remote_code=True,
        # Optional memory controls (tune for your box):
        # max_memory={0: "10GiB", "cpu": "30GiB"},
        # offload_state_dict=True,
        # offload_folder="offload",
    )

    # Optional: choose attention backend
    try:
        model.config.attn_implementation = "flash_attention_2"
    except Exception:
        model.config.attn_implementation = "sdpa"

else:
    # --- CPU fallback: load FP32, then (optional) dynamic quant for Linear ops ---
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    ).eval()

    # Optional CPU quantization (weight-only int8 for Linear layers)
    try:
        from torch.ao.quantization import quantize_dynamic
        model = quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8).eval()
        print("Applied CPU dynamic quantization (int8) to Linear layers.")
    except Exception as e:
        print("CPU dynamic quantization not applied:", e)

model.eval()
print("Model loaded.")
